# 📈 04: Evaluation & Analysis
**GUARDIAN-NLP** | Evaluates the trained model on the test set and generates visualizations.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import yaml

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

LABEL_COLS = ['normal', 'depressive', 'hate_speech', 'violent']
THRESHOLD = config['inference']['threshold']
print('Setup complete.')

## 1. Load Model & Test Data

In [ ]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader
from src.models.bert_classifier import GuardianBERT
from src.models.dataset import ToxicDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model_path = '../models/checkpoints/best_model.pt'
tokenizer_path = '../models/checkpoints/tokenizer/'

if not os.path.exists(model_path):
    print('Model not found. Run training notebook first.')
else:
    tokenizer = BertTokenizer.from_pretrained(tokenizer_path)
    model = GuardianBERT(num_labels=4)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    print(f'Model loaded from {model_path}')

## 2. Run Inference on Test Set

In [ ]:
test_path = '../data/processed/test.csv'

if os.path.exists(test_path) and os.path.exists(model_path):
    test_ds = ToxicDataset(test_path, tokenizer, max_length=128)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            probs = model(input_ids, attention_mask)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(batch['labels'].numpy())
    
    y_true = np.vstack(all_labels)
    y_pred_probs = np.vstack(all_probs)
    y_pred = (y_pred_probs >= THRESHOLD).astype(int)
    print(f'Test set: {len(y_true)} samples')
else:
    print('Test data or model not found.')

## 3. Evaluation Metrics

In [ ]:
if 'y_true' in locals():
    from src.evaluate.metrics import evaluate, print_report
    
    metrics = evaluate(y_true, y_pred_probs, threshold=THRESHOLD)
    
    print('=' * 50)
    print('GUARDIAN-NLP — Test Set Metrics')
    print('=' * 50)
    print(f'Micro F1:     {metrics["micro_f1"]:.4f}  (target > 0.82)')
    print(f'Macro F1:     {metrics["macro_f1"]:.4f}')
    print(f'ROC-AUC:      {metrics["roc_auc"]:.4f}  (target > 0.88)')
    print(f'Hamming Loss: {metrics["hamming_loss"]:.4f}')
    print()
    
    for label in LABEL_COLS:
        f1 = metrics.get(f'{label}_f1', 0)
        prec = metrics.get(f'{label}_precision', 0)
        rec = metrics.get(f'{label}_recall', 0)
        auc = metrics.get(f'{label}_auc', 0)
        print(f'[{label}] F1: {f1:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | AUC: {auc:.3f}')
    
    print_report(y_true, y_pred_probs, threshold=THRESHOLD, save_path='../outputs/reports/classification_report.txt')

## 4. Confusion Matrix per Label

In [ ]:
if 'y_true' in locals():
    from sklearn.metrics import confusion_matrix
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for i, (label, ax) in enumerate(zip(LABEL_COLS, axes)):
        cm = confusion_matrix(y_true[:, i], y_pred[:, i])
        sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='RdYlGn',
                    xticklabels=['Pred 0', 'Pred 1'],
                    yticklabels=['True 0', 'True 1'])
        ax.set_title(f'{label.replace("_", " ").title()}', fontweight='bold')
    
    plt.suptitle('Confusion Matrices per Label (Test Set)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to outputs/visualizations/confusion_matrix.png')

## 5. ROC Curves per Label

In [ ]:
if 'y_true' in locals():
    from sklearn.metrics import roc_curve, auc
    
    colors = ['#2ecc71', '#f39c12', '#e74c3c', '#8e0000']
    fig, ax = plt.subplots(figsize=(10, 7))
    
    for i, (label, color) in enumerate(zip(LABEL_COLS, colors)):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_pred_probs[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2.5,
                label=f'{label.replace("_", " ").title()} (AUC = {roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves per Label (Test Set)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.axhline(y=0.88, color='gray', linestyle=':', alpha=0.7, label='Target AUC=0.88')
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('ROC curves saved.')

## 6. Live Inference Demo

In [ ]:
if os.path.exists(model_path):
    from src.inference.predictor import Predictor
    
    predictor = Predictor(
        model_path=model_path,
        tokenizer_path=tokenizer_path,
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )
    
    test_inputs = [
        "I just can't take it anymore. Everything feels pointless and I'm tired of living.",
        "People like you don't deserve to exist in this world.",
        "You will pay for this. I know where you live.",
        "Had such a wonderful day at the beach with my family!",
        "I'm so tired and hopeless. Nobody cares about me.",
    ]
    
    print('GUARDIAN-NLP Live Predictions\n' + '='*60)
    for text in test_inputs:
        result = predictor.predict(text)
        print(f'Text: {text[:80]}...')
        print(f'  Normal: {result["normal"]:.2%} | Depressive: {result["depressive"]:.2%} |', end=' ')
        print(f'Hate: {result["hate_speech"]:.2%} | Violent: {result["violent"]:.2%}')
        print(f'  Severity: {result["severity"]}  |  Active: {result["active_labels"]}')
        print()